<a href="https://colab.research.google.com/github/rylam11/BUS4-118-Prompt-Engineering/blob/main/Ryan_Lam_Exercise1_Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI

## Goal
Build a multi-step customer support flow where the output of one prompt becomes the input to the next prompt.

## Tools Used
- Google Colab
- Python
- Gemini API key

## Test Case
Customer message: "My package says delivered, but I never received it."

## Prompt Chain
1. Identify the customer's issue
2. Gather missing information
3. Propose a solution
4. Decide whether the issue should be escalated to a human

In [1]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say: Gemini connection successful."
)

print(response.text)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


Gemini connection successful.


In [5]:
import time

def ask_gemini(prompt):
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")

            if attempt < 2:
                print("Trying again...")
                time.sleep(5)

    return "Unable to get a response after 3 attempts."

## Testing and Iteration

During testing, Step 3 repeatedly returned a 503 high-demand error even after the retry logic attempted the request three times.

To improve reliability, I changed the Gemini model from Gemini 3.8 Flash to Gemini 3.5 Flash-Lite while keeping the retry logic.

In [12]:
# Version 2 - Updated after testing

import time

def ask_gemini(prompt):
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")

            if attempt < 2:
                print("Trying again...")
                time.sleep(5)

    return "Unable to get a response after 3 attempts."

In [15]:
customer_message = "My package says delivered, but I never received it."

step1_prompt = f"""
You are a customer support assistant.

Read the customer's message and identify the main issue.

Choose one category:
- Delivery issue
- Refund request
- Damaged item
- Wrong item
- Other

Respond with:
Issue category:
Short explanation:

Customer message: {customer_message}
"""

step1_output = ask_gemini(step1_prompt)

print("STEP 1 - IDENTIFY THE ISSUE")
print(step1_output)

STEP 1 - IDENTIFY THE ISSUE
Issue category: Delivery issue
Short explanation: The customer's tracking status shows the package was delivered, but they have not received it.


In [16]:
step2_prompt = f"""
You are a customer support assistant.

The customer's issue was identified as:

{step1_output}

Based on this issue, identify the most important missing information needed before solving the problem.

Ask for no more than 3 pieces of information.
Keep the questions short, clear, and professional.

Respond with:
1.
2.
3.
"""

step2_output = ask_gemini(step2_prompt)

print("STEP 2 - GATHER MISSING INFORMATION")
print(step2_output)

STEP 2 - GATHER MISSING INFORMATION
1. Can you please confirm your order number?
2. Could you verify the exact shipping address provided at checkout?
3. Have you checked with neighbors or your local delivery office/front desk?


In [20]:
customer_details = """
Order number: 12345
Tracking status: Delivered
The customer checked with neighbors and around the property but could not find the package.
"""

In [18]:
step3_prompt = f"""
You are a customer support assistant.

The customer's issue was identified as:

{step1_output}

The missing information you requested was:

{step2_output}

The customer then provided these details:

{customer_details}

Based on all of this information, recommend the best next step.

You may suggest actions such as:
- checking with the shipping carrier
- confirming the delivery location
- contacting the supplier
- requesting a replacement or refund if appropriate

Keep the response polite, clear, and concise.

Respond with:
Recommended solution:
Short explanation:
"""

step3_output = ask_gemini(step3_prompt)

print("STEP 3 - PROPOSE A SOLUTION")
print(step3_output)

STEP 3 - PROPOSE A SOLUTION
Recommended solution: Contact the shipping carrier to file a claim for the missing package and reach out to our support team for a replacement or refund if the carrier cannot locate it.

Short explanation: Since you have already checked your property and with your neighbors, and confirmed your shipping address for order #12345, the next best step is to investigate the delivery with the carrier. If they are unable to resolve the issue, we can proceed with issuing a replacement or refund.


In [19]:
step4_prompt = f"""
You are a customer support assistant.

Review the customer's issue and the proposed solution.

Customer issue:
{step1_output}

Customer details:
{customer_details}

Proposed solution:
{step3_output}

Decide whether this issue should be handled automatically or escalated to a human customer support representative.

Escalate if:
- the customer requests a human
- the issue cannot be confidently resolved
- a refund or replacement requires approval
- the issue is complex or sensitive

Respond with:
Decision: Handle automatically OR Escalate
Reason: One short sentence
"""

step4_output = ask_gemini(step4_prompt)

print("STEP 4 - ESCALATION DECISION")
print(step4_output)

STEP 4 - ESCALATION DECISION
Decision: Escalate
Reason: Issuing a refund or replacement requires approval.


## Exercise 1 Summary

The prompt chain successfully classified the customer issue, gathered missing information, proposed a solution, and decided whether the case should be escalated.

During testing, the Gemini API returned temporary 503 errors. I improved the workflow by adding retry logic and switching to a lighter Gemini model for better reliability.

Final result: The issue was escalated because a refund or replacement requires human approval.